<script src="{{site.baseurl}}/assets/js/code-runner-analytics.js"></script>

## Popcorn Hack 1: Check One SFI Part Record

A record is complete when it has a product name, a racing category, a spec number, and an effective date. Auto Racing and Drag Racing are the valid categories. The runner reports the True/False result for each requirement.

In [ ]:
# CODE_RUNNER: Popcorn Hack 1 - Check SFI Part Fields

part = {
    "product_name": "Replacement Flywheels and Clutch Assemblies",
    "category": "Auto Racing",
    "spec_number": "1.1",
    "effective_date": "Nov. 9, 2001"
}

valid_categories = ["Auto Racing", "Drag Racing"]

# One Boolean check per requirement.
has_product_name = part["product_name"].strip() != ""
valid_category = part["category"] in valid_categories
has_spec_number = part["spec_number"].strip() != ""
has_effective_date = part["effective_date"].strip() != ""

print("Has product name:", has_product_name)
print("Valid racing category:", valid_category)
print("Has spec number:", has_spec_number)
print("Has effective date:", has_effective_date)

# Complete only when ALL four checks are True.
record_is_complete = has_product_name and valid_category and has_spec_number and has_effective_date
print("Record is complete:", record_is_complete)

## Popcorn Hack 2: Backend Create Rule

A new record is created only when all required fields are valid **and** its spec number is not already stored. I test it with a valid record, an invalid record, and a duplicate record.

In [ ]:
# CODE_RUNNER: Popcorn Hack 2 - Backend Create Rule

part = {
    "product_name": "Replacement Flywheels and Clutch Assemblies",
    "category": "Auto Racing",
    "spec_number": "1.1",
    "effective_date": "Nov. 9, 2001"
}

valid_categories = ["Auto Racing", "Drag Racing"]
existing_spec_numbers = ["1.1", "2.1"]


def can_create(record):
    fields_valid = (
        record["product_name"].strip() != ""
        and record["category"] in valid_categories
        and record["spec_number"].strip() != ""
        and record["effective_date"].strip() != ""
    )
    duplicate_spec = record["spec_number"] in existing_spec_numbers
    print("  fields_valid:", fields_valid, "| duplicate_spec:", duplicate_spec)
    # Create only when the fields are valid AND the spec is NOT a duplicate.
    return fields_valid and not duplicate_spec


valid_part = {
    "product_name": "Multiple Disc Clutch Assemblies",
    "category": "Drag Racing",
    "spec_number": "1.2",
    "effective_date": "Feb. 9, 2006"
}

invalid_part = {
    "product_name": "Street Flywheel",
    "category": "Street Car",
    "spec_number": "3.1",
    "effective_date": "Jan. 1, 2026"
}

tests = [("valid", valid_part), ("invalid", invalid_part), ("duplicate", part)]

for label, record in tests:
    print("Test (" + label + "):", record["spec_number"], "-", record["product_name"])
    if can_create(record):
        print("  -> CREATE record")
    else:
        print("  -> REJECT record")

## Popcorn Hack 3: SFI Search Filter

A record matches when the query is in its product name **or** equals its spec number. Only records from accepted racing categories are returned. I run the search with more than one query to show it still works when the query changes.

In [ ]:
# CODE_RUNNER: Popcorn Hack 3 - SFI Search Filter

query = "1.2"

records = [
    {
        "product_name": "Replacement Flywheels and Clutch Assemblies",
        "category": "Auto Racing",
        "spec_number": "1.1"
    },
    {
        "product_name": "Multiple Disc Clutch Assemblies",
        "category": "Drag Racing",
        "spec_number": "1.2"
    },
    {
        "product_name": "Racing Flywheel Record",
        "category": "Auto Racing",
        "spec_number": "2.1"
    }
]

valid_categories = ["Auto Racing", "Drag Racing"]


def search(search_query):
    results = []
    for record in records:
        name_match = search_query.lower() in record["product_name"].lower()
        spec_match = search_query == record["spec_number"]
        accepted_category = record["category"] in valid_categories
        # (name OR spec) AND accepted category
        if (name_match or spec_match) and accepted_category:
            results.append(record)
    return results


for test_query in [query, "flywheel", "9.9"]:
    matches = search(test_query)
    print('Search "' + test_query + '":', len(matches), "match(es)")
    for record in matches:
        print("  ", record["spec_number"], "-", record["product_name"])

## Popcorn Hack 4 / Homework: SFI Backend Validator

A reusable validator for candidate records. A record is accepted when it has a product name, an accepted racing category, a spec number, an effective date, and a spec number that is not already stored. Each rejected record prints the reason it failed.

In [ ]:
# CODE_RUNNER: Popcorn Hack 4 / Homework - SFI Backend Validator

existing_spec_numbers = ["1.1", "2.1"]
valid_categories = ["Auto Racing", "Drag Racing"]

test_records = [
    {
        "product_name": "Multiple Disc Clutch Assemblies",
        "category": "Auto Racing",
        "spec_number": "1.2",
        "effective_date": "Feb. 9, 2006"
    },
    {
        "product_name": "Replacement Flywheels",
        "category": "Street Car",
        "spec_number": "3.1",
        "effective_date": "Jan. 1, 2026"
    },
    {
        "product_name": "Existing Flywheel Record",
        "category": "Drag Racing",
        "spec_number": "1.1",
        "effective_date": "Jan. 1, 2026"
    }
]


def validate_record(record):
    has_product_name = record["product_name"].strip() != ""
    valid_category = record["category"] in valid_categories
    has_spec_number = record["spec_number"].strip() != ""
    has_effective_date = record["effective_date"].strip() != ""
    duplicate_spec = record["spec_number"] in existing_spec_numbers

    is_valid = (
        has_product_name
        and valid_category
        and has_spec_number
        and has_effective_date
        and not duplicate_spec
    )

    reasons = []
    if not has_product_name:
        reasons.append("missing product name")
    if not valid_category:
        reasons.append("category '" + record["category"] + "' is not accepted")
    if not has_spec_number:
        reasons.append("missing spec number")
    if not has_effective_date:
        reasons.append("missing effective date")
    if duplicate_spec:
        reasons.append("spec " + record["spec_number"] + " already exists")

    return is_valid, reasons


for record in test_records:
    is_valid, reasons = validate_record(record)
    if is_valid:
        existing_spec_numbers.append(record["spec_number"])  # now stored
        print("ACCEPT RECORD:", record["spec_number"], "-", record["product_name"])
    else:
        print("REJECT RECORD:", record["spec_number"], "-", record["product_name"], "->", ", ".join(reasons))

print()
print("Stored spec numbers:", existing_spec_numbers)